# Person A — Notebook 1 v2.2: ACDC Circuit Discovery (threshold=0.5)
**CS 590NN | Amogh | Apr 19 — threshold sensitivity ablation**

This notebook is a **sensitivity rerun** of Notebook 1 v2.1. It does not
replace v2.1. Both threshold settings (0.3 and 0.5) are reported in the
paper's methodology section as an ACDC threshold sensitivity analysis.

**Motivation:** The Notebook 4 v2 run showed ACDC at threshold=0.3
(avg n_mlp=11) lost to ROME-trace (n_mlp=5) on both edit_success and KL
drift. On the subset of ACDC samples where circuits were already tight
(n_mlp ≤ 8), ACDC was competitive. This notebook raises the threshold to
0.5 to see whether **size-matched** ACDC circuits close the gap.

**Changes vs v2.1:**
1. `threshold = 0.5` (was 0.3). Targets avg n_mlp ≈ 5.
2. `effect_floor = 0.5` (unchanged). Prevents denominator collapse.
3. `set_use_attn_result(True)` (unchanged from v2). Attention hooks are populated.
4. Output filename: `week2_circuit_log_v2.2.json` (preserves v2, v2.1).
5. Summary cell reports the 0.3 vs 0.5 delta so we can see circuit shrinkage.

**Runtime:** ~20 min on A100 / ~10 min on H100. Same as v2.1.

### 1.0 Install
Run once. Runtime restarts automatically — do not re-run after restart.

In [ ]:
import subprocess, sys, os

def pip(args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + args)

pip(["numpy==1.26.4"])
pip(["transformer-lens","transformers","datasets","accelerate","einops","jaxtyping"])

print("Done. Restarting runtime...")
os.kill(os.getpid(), 9)

### 1.1 Imports
Start here after restart.

In [1]:
import torch, json, re, requests
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from transformer_lens import HookedTransformer, ActivationCache

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    free, total = torch.cuda.mem_get_info()
    print(f"Memory : {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

import numpy as np
assert tuple(int(x) for x in np.__version__.split(".")[:2]) < (2,0),     "NumPy >= 2.0 — re-run cell 1.0 and restart."
print(f"NumPy  : {np.__version__}")

torch.manual_seed(42)
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
print("Imports OK")

Device : cuda
GPU    : NVIDIA A100-SXM4-80GB
Memory : 84.6 GB free / 85.1 GB total
NumPy  : 1.26.4
Imports OK


### 1.2 Load Model

In [2]:
MODEL_NAME_PRIMARY  = "Qwen/Qwen3-0.6B"
MODEL_NAME_FALLBACK = "Qwen/Qwen2.5-0.5B"

def load_model(name):
    return HookedTransformer.from_pretrained(
        name,
        center_unembed=True,
        center_writing_weights=False,
        fold_ln=True,
        refactor_factored_attn_matrices=False,
        device=DEVICE,
    )

try:
    model = load_model(MODEL_NAME_PRIMARY)
    MODEL_NAME = MODEL_NAME_PRIMARY
except Exception as e:
    print(f"Primary failed ({e}). Using fallback...")
    model = load_model(MODEL_NAME_FALLBACK)
    MODEL_NAME = MODEL_NAME_FALLBACK

# v2 fix: enable hook_result so ACDC can score attention heads
model.set_use_attn_result(True)

model.eval()
model.tokenizer.pad_token = model.tokenizer.eos_token
free, total = torch.cuda.mem_get_info()
print(f"Loaded : {MODEL_NAME}")
print(f"Config : {model.cfg.n_layers} layers | {model.cfg.n_heads} heads | d_model={model.cfg.d_model}")
print(f"Memory : {free/1e9:.1f} GB free after load")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Loaded pretrained model Qwen/Qwen3-0.6B into HookedTransformer
Loaded : Qwen/Qwen3-0.6B
Config : 28 layers | 16 heads | d_model=1024
Memory : 81.5 GB free after load


### 1.3 Native ACDC Implementation

In [3]:
class NativeACDC:
    """
    ACDC-style threshold-based circuit discovery via activation patching.
    Conmy et al., NeurIPS 2023. Implemented natively via TransformerLens.

    v2.2: threshold raised from 0.3 to 0.5 to produce tighter circuits (~5 MLPs)
    matched in size to ROME-trace's top-K=5 localization.
    """
    def __init__(self, model, threshold=0.5, effect_floor=0.5):
        self.model        = model
        self.threshold    = threshold
        self.effect_floor = effect_floor

    def _logit_diff(self, logits, target_id, baseline_id):
        return (logits[0, -1, target_id] - logits[0, -1, baseline_id]).item()

    def run(self, prompt, target_true, target_new):
        true_id = self.model.to_tokens(target_true, prepend_bos=False)[0, 0].item()
        new_id  = self.model.to_tokens(target_new,  prepend_bos=False)[0, 0].item()
        tokens  = self.model.to_tokens(prompt)

        corrupt = tokens.clone()
        if tokens.shape[1] > 2:
            corrupt[0, 1:-1] = torch.randint(1000, self.model.cfg.d_vocab-1, (tokens.shape[1]-2,))

        with torch.no_grad():
            clean_logits,   clean_cache   = self.model.run_with_cache(tokens)
            corrupt_logits, corrupt_cache = self.model.run_with_cache(corrupt)

        clean_score   = self._logit_diff(clean_logits,   true_id, new_id)
        corrupt_score = self._logit_diff(corrupt_logits, true_id, new_id)
        raw_effect    = abs(clean_score - corrupt_score)
        effect_range  = max(raw_effect, self.effect_floor)
        effect_clamped = raw_effect < self.effect_floor

        causal_scores = {}

        for layer in range(self.model.cfg.n_layers):
            # Attention heads
            hn = f"blocks.{layer}.attn.hook_result"
            if hn in clean_cache:
                for head in range(self.model.cfg.n_heads):
                    def make_attn(h=head, n=hn):
                        ca = corrupt_cache[n][:, :, h:h+1, :].clone()
                        def fn(v, hook): v[:, :, h:h+1, :] = ca; return v
                        return fn
                    with torch.no_grad():
                        pl = self.model.run_with_hooks(tokens, fwd_hooks=[(hn, make_attn())])
                    causal_scores[f"attn_{layer}_{head}"] = abs(self._logit_diff(pl, true_id, new_id) - clean_score) / effect_range

            # MLP layers
            hn = f"blocks.{layer}.hook_mlp_out"
            if hn in clean_cache:
                def make_mlp(n=hn):
                    ca = corrupt_cache[n].clone()
                    def fn(v, hook): return ca
                    return fn
                with torch.no_grad():
                    pl = self.model.run_with_hooks(tokens, fwd_hooks=[(hn, make_mlp())])
                causal_scores[f"mlp_{layer}"] = abs(self._logit_diff(pl, true_id, new_id) - clean_score) / effect_range

        # Free caches immediately
        del clean_cache, corrupt_cache, clean_logits, corrupt_logits
        torch.cuda.empty_cache()

        attn_heads, mlp_layers = [], []
        for node, score in causal_scores.items():
            if score >= self.threshold:
                parts = node.split("_")
                if parts[0] == "attn": attn_heads.append((int(parts[1]), int(parts[2])))
                else:                  mlp_layers.append(int(parts[1]))

        return {
            "attn_heads":     sorted(set(attn_heads)),
            "mlp_layers":     sorted(set(mlp_layers)),
            "causal_scores":  causal_scores,
            "clean_score":    clean_score,
            "corrupt_score":  corrupt_score,
            "raw_effect":     raw_effect,
            "effect_clamped": effect_clamped,
        }

print("NativeACDC (v2.2) defined. Threshold=0.5, effect_floor=0.5.")


NativeACDC (v2.2) defined. Threshold=0.5, effect_floor=0.5.


### 1.4 Load CounterFact

In [4]:
@dataclass
class EditSample:
    prompt:          str
    target_new:      str
    target_true:     str
    related_prompts: List[str] = field(default_factory=list)

print("Downloading CounterFact from ROME project source...")
raw_data = requests.get("https://rome.baulab.info/data/dsets/counterfact.json", timeout=60).json()
print(f"Downloaded {len(raw_data)} records")

N_SAMPLES = 50

def parse_counterfact(raw):
    neighborhood = raw.get("neighborhood_prompts", [])
    related = [p for p in neighborhood[:3] if isinstance(p, str)]
    return EditSample(
        prompt=raw["requested_rewrite"]["prompt"].format(raw["requested_rewrite"]["subject"]),
        target_new=" "  + raw["requested_rewrite"]["target_new"]["str"],
        target_true=" " + raw["requested_rewrite"]["target_true"]["str"],
        related_prompts=related,
    )

cf_samples = [parse_counterfact(raw_data[i]) for i in range(N_SAMPLES)]
print(f"Loaded {N_SAMPLES} samples")
print(f"Example: '{cf_samples[0].prompt}' -> true='{cf_samples[0].target_true}'")

Downloaded 21919 records
Loaded 50 samples
Example: 'The mother tongue of Danielle Darrieux is' -> true=' French'


### 1.5 Run ACDC on 50 Samples

In [5]:
acdc        = NativeACDC(model, threshold=0.5, effect_floor=0.5)
circuit_log = []

print(f"Running ACDC v2.2 on {N_SAMPLES} samples...")
print(f"  threshold    = {acdc.threshold}  (was 0.3 in v2.1)")
print(f"  effect_floor = {acdc.effect_floor}")
print("Caches freed after each sample.\n")

for i, s in enumerate(cf_samples):
    result = acdc.run(s.prompt, s.target_true, s.target_new)
    circuit_log.append({
        "idx":             i,
        "prompt":          s.prompt,
        "attn_heads":      result["attn_heads"],
        "mlp_layers":      result["mlp_layers"],
        "clean_score":     round(result["clean_score"],    4),
        "corrupt_score":   round(result["corrupt_score"],  4),
        "raw_effect":      round(result["raw_effect"],     4),
        "effect_clamped":  bool(result["effect_clamped"]),
        "n_attn":          len(result["attn_heads"]),
        "n_mlp":           len(result["mlp_layers"]),
    })
    if i % 10 == 0:
        free = torch.cuda.mem_get_info()[0] / 1e9
        clamp = " [CLAMPED]" if result["effect_clamped"] else ""
        print(f"  [{i+1:>2}/{N_SAMPLES}]  n_attn={len(result['attn_heads']):>3}  "
              f"n_mlp={len(result['mlp_layers']):>2}  "
              f"raw_eff={result['raw_effect']:.2f}{clamp}  gpu_free={free:.1f}GB")

with open(RESULTS_DIR / "week2_circuit_log_v2.2.json", "w") as f:
    json.dump(circuit_log, f, indent=2)

from collections import Counter
all_attn = [str(h) for e in circuit_log for h in e["attn_heads"]]
all_mlp  = [str(l) for e in circuit_log for l in e["mlp_layers"]]
print(f"\nTop attention heads : {Counter(all_attn).most_common(5)}")
print(f"Top MLP layers      : {Counter(all_mlp).most_common(5)}")
print(f"Saved -> {RESULTS_DIR}/week2_circuit_log_v2.2.json")


Running ACDC v2.2 on 50 samples...
  threshold    = 0.5  (was 0.3 in v2.1)
  effect_floor = 0.5
Caches freed after each sample.

  [ 1/50]  n_attn=  2  n_mlp= 1  raw_eff=3.58  gpu_free=81.3GB
  [11/50]  n_attn=  0  n_mlp= 0  raw_eff=4.38  gpu_free=81.3GB
  [21/50]  n_attn= 21  n_mlp= 9  raw_eff=0.16 [CLAMPED]  gpu_free=81.3GB
  [31/50]  n_attn= 46  n_mlp=20  raw_eff=0.02 [CLAMPED]  gpu_free=81.3GB
  [41/50]  n_attn=  5  n_mlp= 7  raw_eff=2.88  gpu_free=81.3GB

Top attention heads : [('(0, 3)', 25), ('(0, 12)', 20), ('(0, 7)', 16), ('(0, 14)', 13), ('(0, 15)', 13)]
Top MLP layers      : [('0', 33), ('3', 25), ('1', 22), ('2', 22), ('4', 18)]
Saved -> results/week2_circuit_log_v2.2.json


### 1.6 Save and Download Results

In [6]:
import shutil
from datetime import datetime
from collections import Counter

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Core circuit size stats
n_attn_list = [e["n_attn"] for e in circuit_log]
n_mlp_list  = [e["n_mlp"]  for e in circuit_log]
n_clamped   = sum(1 for e in circuit_log if e["effect_clamped"])

# Distribution
buckets_def = [("<=2", 2), ("3-5", 5), ("6-10", 10), ("11-20", 20), (">20", 10**9)]
dist = {}
prev = 0
for label, upper in buckets_def:
    dist[label] = sum(1 for n in n_mlp_list if prev < n <= upper)
    prev = upper

# Sanity check: effect-range / n_attn correlation
def pearson(xs, ys):
    mx, my = sum(xs)/len(xs), sum(ys)/len(ys)
    num = sum((x-mx)*(y-my) for x,y in zip(xs, ys))
    den = (sum((x-mx)**2 for x in xs) * sum((y-my)**2 for y in ys)) ** 0.5
    return num/den if den else 0.0

effect_vs_nattn = pearson([e["raw_effect"] for e in circuit_log], n_attn_list)

# v2.1 comparison numbers (from Notebook 1 v2.1 run, hardcoded for reference)
V21_AVG_ATTN = 21.4
V21_AVG_MLP  = 10.96
V21_MEDIAN_ATTN = 8
V21_MEDIAN_MLP  = 10
V21_MAX_ATTN = 111

summary = {
    "author":           "Amogh",
    "notebook":         "Notebook 1 v2.2 — ACDC Circuit Discovery (threshold=0.5)",
    "model":            MODEL_NAME,
    "timestamp":        timestamp,
    "n_samples":        N_SAMPLES,
    "threshold":        acdc.threshold,
    "effect_floor":     acdc.effect_floor,
    "is_sensitivity_ablation": True,
    "complements":      "week2_circuit_log_v2.1.json (threshold=0.3)",
    "circuit_summary": {
        "avg_attn_per_sample": round(sum(n_attn_list) / N_SAMPLES, 2),
        "avg_mlp_per_sample":  round(sum(n_mlp_list)  / N_SAMPLES, 2),
        "median_n_attn":       sorted(n_attn_list)[N_SAMPLES//2],
        "median_n_mlp":        sorted(n_mlp_list)[N_SAMPLES//2],
        "max_n_attn":          max(n_attn_list),
        "max_n_mlp":           max(n_mlp_list),
        "top_attn_heads":      Counter(all_attn).most_common(5),
        "top_mlp_layers":      Counter(all_mlp).most_common(5),
        "mlp_size_distribution": dist,
        "n_samples_effect_clamped":  n_clamped,
        "effect_vs_nattn_pearson":   round(effect_vs_nattn, 3),
    },
    "comparison_vs_v21": {
        "avg_attn":  {"v2.1": V21_AVG_ATTN,  "v2.2": round(sum(n_attn_list)/N_SAMPLES, 2)},
        "avg_mlp":   {"v2.1": V21_AVG_MLP,   "v2.2": round(sum(n_mlp_list)/N_SAMPLES, 2)},
        "median_mlp": {"v2.1": V21_MEDIAN_MLP, "v2.2": sorted(n_mlp_list)[N_SAMPLES//2]},
    }
}

with open(RESULTS_DIR / f"summary_nb1v2.2_{timestamp}.json", "w") as f:
    json.dump(summary, f, indent=2)

zip_path = f"/content/PersonA_Notebook1v2.2_results_{timestamp}"
shutil.make_archive(zip_path, "zip", RESULTS_DIR)

print("=" * 70)
print(f"  NOTEBOOK 1 v2.2 RESULTS — Amogh (threshold sensitivity ablation)")
print(f"  {timestamp}")
print("=" * 70)
print(f"  Config              : threshold={acdc.threshold}  effect_floor={acdc.effect_floor}")
print()
print(f"  {'metric':<24} {'v2.1 (thr=0.3)':>16} {'v2.2 (thr=0.5)':>16} {'delta':>9}")
print("  " + "-" * 66)
print(f"  {'avg n_attn':<24} {V21_AVG_ATTN:>16.2f} "
      f"{summary['circuit_summary']['avg_attn_per_sample']:>16.2f} "
      f"{summary['circuit_summary']['avg_attn_per_sample'] - V21_AVG_ATTN:>+9.2f}")
print(f"  {'avg n_mlp':<24} {V21_AVG_MLP:>16.2f} "
      f"{summary['circuit_summary']['avg_mlp_per_sample']:>16.2f} "
      f"{summary['circuit_summary']['avg_mlp_per_sample'] - V21_AVG_MLP:>+9.2f}")
print(f"  {'median n_mlp':<24} {V21_MEDIAN_MLP:>16} "
      f"{summary['circuit_summary']['median_n_mlp']:>16} "
      f"{summary['circuit_summary']['median_n_mlp'] - V21_MEDIAN_MLP:>+9}")
print(f"  {'max n_attn':<24} {V21_MAX_ATTN:>16} "
      f"{summary['circuit_summary']['max_n_attn']:>16} "
      f"{summary['circuit_summary']['max_n_attn'] - V21_MAX_ATTN:>+9}")
print()
print(f"  Samples effect-clamped : {n_clamped} / {N_SAMPLES}")
print(f"  Pearson(effect, n_attn): {effect_vs_nattn:+.3f}")
print()
print("  n_mlp distribution (v2.2):")
for k, v in dist.items():
    bar = "#" * v
    print(f"    {k:<6} : {v:>2}  {bar}")
print()
print("  Top heads           :", summary["circuit_summary"]["top_attn_heads"])
print()
print("  Files saved:")
for fp in sorted(RESULTS_DIR.glob("*.json")):
    print(f"    {fp.name:<44}  {fp.stat().st_size//1024:>4} KB")
print(f"\n  Download: {zip_path}.zip")
print("=" * 70)

# Sanity guidance
avg_mlp_v22 = summary['circuit_summary']['avg_mlp_per_sample']
if avg_mlp_v22 > 7:
    print("\n  NOTE: avg n_mlp > 7. Threshold=0.5 still too loose. Consider 0.7.")
elif avg_mlp_v22 < 2:
    print("\n  NOTE: avg n_mlp < 2. Threshold=0.5 may be too strict. Consider 0.4.")
else:
    print(f"\n  Circuit size now comparable to ROME-trace's top-K=5 (avg n_mlp={avg_mlp_v22:.1f}).")
    print("  Upload week2_circuit_log_v2.2.json to Notebook 4 v2 for fair comparison.")

from google.colab import files
files.download(f"{zip_path}.zip")


  NOTEBOOK 1 v2.2 RESULTS — Amogh (threshold sensitivity ablation)
  20260419_021646
  Config              : threshold=0.5  effect_floor=0.5

  metric                     v2.1 (thr=0.3)   v2.2 (thr=0.5)     delta
  ------------------------------------------------------------------
  avg n_attn                          21.40             9.22    -12.18
  avg n_mlp                           10.96             6.52     -4.44
  median n_mlp                           10                4        -6
  max n_attn                            111               66       -45

  Samples effect-clamped : 5 / 50
  Pearson(effect, n_attn): -0.516

  n_mlp distribution (v2.2):
    <=2    : 13  #############
    3-5    : 12  ############
    6-10   :  5  #####
    11-20  : 12  ############
    >20    :  1  #

  Top heads           : [('(0, 3)', 25), ('(0, 12)', 20), ('(0, 7)', 16), ('(0, 14)', 13), ('(0, 15)', 13)]

  Files saved:
    summary_nb1v2.2_20260419_021646.json             1 KB
    week2_circuit_l

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>